# Olist Seller Intelligence Report

This notebook presents an analysis of the [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data), a real-world e-commerce dataset covering orders placed between 2016 and 2018 across multiple Brazilian marketplaces.

The analysis is framed as a data consultancy project: Olist's commercial team wants to understand how sellers are performing, what's driving poor customer reviews, and where inefficiencies exist in their logistics. The goal is to produce findings and recommendations that could inform real operational decisions.

The dataset is stored in MongoDB Atlas and queried using PyMongo. Aggregation pipelines are the primary analytical tool, with results passed into Pandas dataframes for presentation and visualisation. The secondary aim of the notebook is to provide a working demonstration of MongoDB's aggregation framework, index optimisation, and Python driver usage.

# Section 1 - Setup and Data Quality Audit

## Tasks

The client has asked me to first verify that the data is complete and clean before any analysis. I also need to log the session and flag any known data issues for the record.

### 1.1 Connect to the database and list all collections

In [1]:
#import libraries
from pymongo import MongoClient
import pandas as pd
import os
from dotenv import load_dotenv
from pprint import pprint
import datetime

In [2]:
#read constants and connect to database
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["oilist-sales"]

In [3]:
#store collections as variable and view
collections = db.list_collection_names()
print(collections)

['products', 'order-payments', 'geolocation', 'order-reviews', 'category-translation', 'order-items', 'customers', 'orders', 'sellers']


All collections are present as expected.

### 1.2. Inspect one document from each collection

In [4]:
for collection in collections:
    doc = db[collection].find_one()
    schema = pd.DataFrame(
        [(field, type(value).__name__) for field, value in doc.items()],
        columns=["field", "type"]
    )
    print(collection)
    display(schema)

products


,field,type
0,_id,ObjectId
1,product_id,str
2,product_category_name,str
3,product_name_lenght,int
4,product_description_lenght,int
5,product_photos_qty,int
6,product_weight_g,int
7,product_length_cm,int
8,product_height_cm,int
9,product_width_cm,int


order-payments


,field,type
0,_id,ObjectId
1,order_id,str
2,payment_sequential,int
3,payment_type,str
4,payment_installments,int
5,payment_value,Decimal128


geolocation


...

,field,type
0,_id,ObjectId
1,customer_id,str
2,customer_unique_id,str
3,customer_zip_code_prefix,str
4,customer_city,str
5,customer_state,str


orders


,field,type
0,_id,ObjectId
1,order_id,str
2,customer_id,str
3,order_status,str
4,order_purchase_timestamp,str
5,order_approved_at,str
6,order_delivered_carrier_date,str
7,order_delivered_customer_date,str
8,order_estimated_delivery_date,datetime


sellers


,field,type
0,_id,ObjectId
1,seller_id,str
2,seller_zip_code_prefix,str
3,seller_city,str
4,seller_state,str


### 1.3. Create a collection (with a validation schema) to record data audits

In [9]:
# db.create_collection("data_audit", validator = {"$jsonSchema": {
#     "required": ["auditor", "date", "details"],
#     "properties": {
#         "auditor": {"bsonType": "string"},
#         "date": {"bsonType": "date"},
#         "details": {"bsonType": "string"}
#     }
# }})

Collection(Database(MongoClient(host=['ac-vf0ykm7-shard-00-02.musfbjl.mongodb.net:27017', 'ac-vf0ykm7-shard-00-00.musfbjl.mongodb.net:27017', 'ac-vf0ykm7-shard-00-01.musfbjl.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='liamscluster', authsource='admin', replicaset='atlas-1435mb-shard-0', tls=True), 'oilist-sales'), 'data_audit')

In [10]:
#add audit log document to collection
# db.data_audit.insert_one(
#     {
#         "auditor": "Liam Cottrell",
#         "date": datetime.datetime.now(),
#         "details": "Created data audit collection and viewed schemas for each collection"
#     }
# )

InsertOneResult(ObjectId('6a287866482bcd769a72ecd2'), acknowledged=True)

In [11]:
#view document to confirm successful creation
db.data_audit.find_one()

{'_id': ObjectId('6a287866482bcd769a72ecd2'),
 'auditor': 'Liam Cottrell',
 'date': datetime.datetime(2026, 6, 9, 21, 32, 38, 894000),
 'details': 'Created data audit collection and viewed schemas for each collection'}

### 1.4 Count documents in each collection and present the results